In [58]:
import os
import json
import pandas as pd
import traceback

In [59]:
from langchain.chat_models import ChatOpenAI

In [60]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

True

In [61]:
KEY=os.getenv("OPENAI_API_KEY")

In [62]:
llm=ChatOpenAI(openai_api_key=KEY, model_name="gpt-4o-mini", temperature=0.5)

In [63]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
# from langchain_community.callbacks.manager import get_openai_callback
import PyPDF2

In [64]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [65]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [66]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [67]:
quiz_chain=LLMChain(llm=llm,prompt=quiz_generation_prompt,output_key="quiz",verbose=True)

In [68]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [69]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject","quiz"], template=TEMPLATE)

In [70]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [71]:
generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [72]:
file_path=r"/workspaces/mcqgen/data.txt"
file_path

'/workspaces/mcqgen/data.txt'

In [73]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [74]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [75]:
NUMBER=15 
SUBJECT="Galaxy Z Flip6"
TONE="Medium"

In [76]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text:What's new and different about the <Galaxy Z Flip6> ? 	"Would you believe it if I told you there's a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!

The Z Flip6's 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.
Additionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy


> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

Text:What's new and different about the <Galaxy Z Flip6> ? 	"Would you believe it if I told you there's a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!

The Z Flip6's 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.
Additionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy more vivid LED effects

In [77]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost:.3f}")

Total Tokens:5617
Prompt Tokens:3174
Completion Tokens:2443
Total Cost:0.000


In [78]:
response

{'text': 'What\'s new and different about the <Galaxy Z Flip6> ? \t"Would you believe it if I told you there\'s a way to stand out on selfies, customization and even communication? The Galaxy Z Flip6 offers all of this with Galaxy AI!\n\nThe Z Flip6\'s 50 MP flagship camera now provides you with incredible portraits, even at nights. You can get the best angle using automatic AI zoom without touching anything, even from a distance. Furthermore, Galaxy AI lets you edit your favorite memories even better by generating portraits in various styles, and adding 3D effects to images for an immersive viewing experience. AI experience still continues on every screen you see. You can tap or draw on the text to use further AI actions, which can let you compose a new image by adding sketches.\nAdditionally, you can even create unique images or try out things like interactive wallpapers. You can also can enjoy more vivid LED effects on the screen with interactive motion content by attaching LED Effe

In [79]:
quiz = response.get('quiz')
quiz = json.loads(quiz)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [ ]:
quiz_table_data

[{'MCQ': "What is a key feature of the Galaxy Z Flip6's camera?",
  'Choices': 'a: 50 MP flagship camera | b: 20 MP camera | c: 10 MP camera | d: No camera',
  'Correct': 'a'},
 {'MCQ': 'What unique feature does the Z Flip6 offer for selfies?',
  'Choices': 'a: AI zoom | b: Manual zoom | c: No zoom | d: Digital zoom only',
  'Correct': 'a'},
 {'MCQ': 'Which colors is the Galaxy Z Flip6 available in?',
  'Choices': 'a: Red, Green, Blue | b: Blue, Silver Shadow, Mint | c: Black, White, Yellow | d: Purple, Orange, Pink',
  'Correct': 'b'},
 {'MCQ': 'What exclusive colors can be found on Samsung.com?',
  'Choices': 'a: Crafted Black, White, Peach | b: Blue, Mint, Silver | c: Red, Yellow, Green | d: Pink, Grey, Brown',
  'Correct': 'a'},
 {'MCQ': 'What is a new feature of the Camcorder grip in the Z Flip6?',
  'Choices': 'a: Voice activation | b: Zoom Rocker | c: Touch screen | d: Flashlight',
  'Correct': 'b'},
 {'MCQ': "What material is used for the Z Flip6's cases?",
  'Choices': 'a: Pla

In [ ]:
quiz = pd.DataFrame(quiz_table_data)

In [ ]:
quiz.to_csv("GalaxyZFlip6.csv",index=False)